# 16 — LGBM Hyperparameter Search with Optuna

All LGBM notebooks (07, 11, 15) used fixed hyperparameters selected by intuition:
```
n_estimators=1000, num_leaves=64, learning_rate=0.05,
subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, min_child_samples=10
```
This notebook uses Optuna to search the hyperparameter space with scaffold 5-fold CV as the
objective, targeting the LGBM_aug configuration (with null feature + upsampling).

**Motivation**: num_leaves=64 was never validated — the dataset has 4,139 points, and optimal
tree depth could be much larger or smaller. A 50-trial search takes ~40 min and could push
OOF RAE from 0.5633 to ~0.54.

**Runtime**: ~40–60 min (50 trials × 5 folds × ~30s/fold).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pxr.data import load_train, load_test, load_counter
from pxr.chem import to_inchikey, bemis_murcko, morgan_fp_batch
from pxr.featurize import combined, impute
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.preprocess import upsample_by_category
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
SEED = 42
N_TRIALS = 60
print(f"optuna {optuna.__version__}  |  n_trials={N_TRIALS}")

In [ ]:
# ── 2. Load data and precompute features ─────────────────────────────────────
train = load_train()
counter = load_counter()
te = load_test()

smiles_tr = train['smiles'].tolist()
y_tr      = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=5, seed=SEED)

print("Computing combined features ...")
X_base   = impute(combined(smiles_tr))            # (4139, 2265)
X_te_base = impute(combined(te['smiles'].tolist()))

# Null feature: counter-assay pEC50 — median per inchikey to avoid merge duplication
tr_ik  = train.assign(inchikey=train['smiles'].map(to_inchikey))
ct_ik  = counter.assign(inchikey=counter['smiles'].map(to_inchikey))
ct_null = ct_ik.groupby('inchikey', as_index=False)['pec50'].median().rename(columns={'pec50': 'pec50_null'})
joined  = tr_ik.merge(ct_null[['inchikey', 'pec50_null']], on='inchikey', how='left')
assert len(joined) == len(train)
pec50_null = joined['pec50_null'].values.copy()

# Impute null feature on full dataset (simple: median of known values)
# Per-fold imputation would leak; use a single global median for Optuna speed
null_median = np.nanmedian(pec50_null)
null_full   = np.where(np.isnan(pec50_null), null_median, pec50_null)

X_aug    = np.hstack([X_base,    null_full.reshape(-1, 1)])    # (4139, 2266)
X_te_aug = np.hstack([X_te_base, np.full((len(te), 1), null_median)])

print(f"Train: {X_aug.shape}  |  Test: {X_te_aug.shape}")
print(f"Null coverage: {(~np.isnan(pec50_null)).mean():.1%}  (median={null_median:.3f})")

In [ ]:
# ── 3. Optuna objective ───────────────────────────────────────────────────────
def objective(trial: optuna.Trial) -> float:
    params = dict(
        n_estimators     = 2000,
        num_leaves       = trial.suggest_int('num_leaves', 32, 256),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        min_child_samples= trial.suggest_int('min_child_samples', 5, 60),
        n_jobs=4, verbose=-1,
    )
    oof = np.full(len(y_tr), np.nan)
    for tr_idx, va_idx in splits:
        X_tr_up, y_tr_up = upsample_by_category(X_aug[tr_idx], y_tr[tr_idx])
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr_up, y_tr_up,
              eval_set=[(X_aug[va_idx], y_tr[va_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(X_aug[va_idx])
    return rae_fn(y_tr, oof)

# Reference: current fixed-param performance
ref_params = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1,
)
oof_ref = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    X_tr_up, y_tr_up = upsample_by_category(X_aug[tr_idx], y_tr[tr_idx])
    m = lgb.LGBMRegressor(**ref_params)
    m.fit(X_tr_up, y_tr_up)
    oof_ref[va_idx] = m.predict(X_aug[va_idx])
print(f"Reference (fixed params, global-median null) OOF RAE: {rae_fn(y_tr, oof_ref):.4f}")

In [ ]:
# ── 4. Run Optuna study ───────────────────────────────────────────────────────
import time
sampler = optuna.samplers.TPESampler(seed=SEED)
study   = optuna.create_study(direction='minimize', sampler=sampler)

t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
elapsed = time.time() - t0

print(f"\nOptuna search complete: {N_TRIALS} trials in {elapsed/60:.1f} min")
print(f"Best OOF RAE: {study.best_value:.4f}")
print(f"Best params:  {study.best_params}")

In [ ]:
# ── 5. Optuna visualisation ───────────────────────────────────────────────────
trials_df = study.trials_dataframe()
print(f"Top-5 trials:")
print(trials_df.sort_values('value').head(5)[['number','value','params_num_leaves',
                                               'params_learning_rate',
                                               'params_min_child_samples']].to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(trials_df['number'], trials_df['value'], 'o', alpha=0.5, ms=4, color='steelblue')
axes[0].axhline(study.best_value, color='red', ls='--', lw=1.5, label=f'best={study.best_value:.4f}')
axes[0].axhline(rae_fn(y_tr, oof_ref), color='gray', ls=':', lw=1.5, label='fixed params')
axes[0].set(xlabel='Trial', ylabel='OOF RAE', title='Optuna optimisation history')
axes[0].legend()
axes[1].scatter(trials_df['params_num_leaves'], trials_df['value'],
                c=trials_df['params_learning_rate'], cmap='viridis', alpha=0.7, s=30)
axes[1].set(xlabel='num_leaves', ylabel='OOF RAE', title='num_leaves vs RAE (colour=lr)')
plt.colorbar(axes[1].collections[0], ax=axes[1], label='learning_rate')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'figures' / '16_optuna.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 6. Retrain with best params + compute honest OOF ─────────────────────────
best_p = dict(
    n_estimators=2000,
    n_jobs=4, verbose=-1,
    **study.best_params
)

oof_tuned = np.full(len(y_tr), np.nan)
for tr_idx, va_idx in splits:
    X_tr_up, y_tr_up = upsample_by_category(X_aug[tr_idx], y_tr[tr_idx])
    m = lgb.LGBMRegressor(**best_p)
    m.fit(X_tr_up, y_tr_up,
          eval_set=[(X_aug[va_idx], y_tr[va_idx])],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(-1)])
    oof_tuned[va_idx] = m.predict(X_aug[va_idx])

tuned_rae = rae_fn(y_tr, oof_tuned)
print(f"Reference (fixed params) OOF RAE: {rae_fn(y_tr, oof_ref):.4f}")
print(f"Tuned LGBM_aug OOF RAE:          {tuned_rae:.4f}")
print(f"Improvement: {rae_fn(y_tr, oof_ref) - tuned_rae:.4f}")

np.save(DATA_PROCESSED / 'oof_lgbm_tuned.npy', oof_tuned)

In [ ]:
# ── 7. Train final model on full data + predict test ─────────────────────────
X_tr_up_f, y_tr_up_f = upsample_by_category(X_aug, y_tr)
final_m = lgb.LGBMRegressor(**best_p)
final_m.fit(X_tr_up_f, y_tr_up_f)
te_tuned = final_m.predict(X_te_aug)
te_tuned = np.clip(te_tuned, y_tr.min() - 0.5, y_tr.max() + 0.5)

np.save(DATA_PROCESSED / 'te_lgbm_tuned.npy', te_tuned)

# Blend with Chemprop via inverse-RAE
for fname in ['10_expanded_multitask.csv', '08_chemprop_cv_blend.csv']:
    cp_path = SUBMISSIONS / fname
    if cp_path.exists():
        cp_preds = pd.read_csv(cp_path).set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values
        cp_src   = fname; break

chemprop_rae = 0.5736
w_cp   = (1/chemprop_rae) / (1/chemprop_rae + 1/tuned_rae)
final_preds = w_cp * cp_preds + (1 - w_cp) * te_tuned
final_preds = np.clip(final_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)
print(f"Chemprop weight: {w_cp:.3f}  |  Tuned LGBM weight: {1-w_cp:.3f}")

In [ ]:
# ── 8. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values,
                    'SMILES':        te['smiles'].values,
                    'pEC50':         final_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '16_lgbm_optuna.csv'
sub.to_csv(out, index=False)

print(f"Saved: {out}")
print(f"Tuned LGBM OOF RAE:     {tuned_rae:.4f}")
print(f"Grand-15 OOF RAE:       0.5473")
print(f"Best to date:           {min(tuned_rae, 0.5473):.4f}")
print(sub['pEC50'].describe().round(3))